In [ ]:
from pathlib import Path
from pypdf import PdfReader
import sys
sys.path.append(str(Path.cwd().parent))
from src.chunker import clean_text, chunker

In [ ]:
pdf = PdfReader(Path("../data/samplerag.pdf"))
raw_pages=[]
for i,page in enumerate(pdf.pages):
    text = page.extract_text()
    if text:
        raw_pages.append(text)
    else:
        print(f"{i+1} has no text")
raw_text="\n".join(raw_pages)
cleaned_te = clean_text(raw_text)
chunks= chunker(cleaned_te,500,100)
print(chunks[0])

In [ ]:
chunk_metadata = []
for i,chunk in enumerate(chunks):
    chunk_metadata.append({"chunk_id":i, "text":chunk,
                           "source":"samplerag.pdf","length":len(chunk)})
chunk_metadata[0]

In [ ]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
chunk_embeddings = embedding_model.encode(chunks,show_progress_bar=True,convert_to_numpy=True)

print(f"Embedding matrix shape (chunks,dimensions):{chunk_embeddings.shape} ")

In [ ]:
import faiss
import numpy as np

In [ ]:
###faiss usually wants float-32

chunk_embeddings = np.array(chunk_embeddings).astype("float32")
faiss.normalize_L2(chunk_embeddings)

embedding_dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(chunk_embeddings)
print(f"Number of vectors in the index:{index.ntotal}")

In [ ]:
query="What is rag?"

query_embedding = embedding_model.encode([query],convert_to_numpy=True).astype("float32")
faiss.normalize_L2(query_embedding)

print(query_embedding.shape)

In [ ]:
top_k=5

scores, indices = index.search(query_embedding,top_k)
print("Scores",scores)
print("Indices:", indices)

In [ ]:
for rank, chunk_idx in enumerate(indices[0],start=1):
    print(f"\n---Rank {rank} | Chunk ID: {chunk_idx}---")
    print(chunk_metadata[chunk_idx]["text"])

In [ ]:
questions=["what is ingestion?","what is vector database?","what is reranking?","what are limitations of rag"]

questions_embeddings = embedding_model.encode(questions,convert_to_numpy=True).astype("float32")
faiss.normalize_L2(questions_embeddings)
print(questions_embeddings.shape)

top_k=3
scores,indices = index.search(questions_embeddings,top_k)
print(indices.shape)

for i in range(indices.shape[0]):
    print(f"\n{questions[i]}")
    for rank, chunk_idx in enumerate(indices[i],start=1):
        print(f"\n--------rank: {rank} || chunk-id:{chunk_idx}-----")
        print(f"\n{chunk_metadata[chunk_idx]["text"]}")

# Generation

In [ ]:
def retrieve_chunks(query,top_k=3):
    qembed= embedding_model.encode([query],convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(qembed)
    scores,indices = index.search(qembed,top_k)
    results=[]
    for score, chunk_idx in zip(scores[0],indices[0]):
        results.append({
            "chunk_id":int(chunk_idx),
            "score":float(score),
            "text":chunk_metadata[chunk_idx]["text"],
            "source":chunk_metadata[chunk_idx]["source"]
        })
    return results

In [ ]:
res = retrieve_chunks("What is RAG?",3)

for r in res:
    print("\n")
    print(f"chunk_ id:{r["chunk_id"]}")
    print(f"Score:{r["score"]}")
    print(f"\n{r["text"]}")

In [ ]:
#Augumentation: Taking the retrieved chunks and inserting them into prompt as external context.

def build_prompt(query, retrieve_chunks):
    context_blocks=[]
    for i, chunk in enumerate(retrieve_chunks,start=1):
        context_blocks.append(
            f"[Context {i} | Chunk ID: {chunk["chunk_id"]} | Score: {chunk["score"]}]\n{chunk["text"]}"
        )

    context_text='\n'.join(context_blocks)
    prompt = f"""
You are a helpful assistant answering questions using only the provided context.

Instructions:
- Use the context below to answer the user's question.
- If the answer is not clearly supported by the context, say: "I don't know based on the provided context."
- Keep the answer clear and concise.
- Do not invent facts outside the context.

Retrieved Context:{context_text}
User Question:{query}
Answer:""".strip()

    return prompt
    




In [ ]:
prompt = build_prompt("What is RAG?",res)

print(prompt[:3000])

In [ ]:
from dotenv import load_dotenv
import os
import google.genai as genai

load_dotenv()

api_key=os.getenv("GEMINI_API_KEY")
client= genai.Client(api_key=api_key)


In [ ]:
def generate_answer_with_gemini(prompt):
    response = client.models.generate_content(model="gemini-2.0-flash",contents=prompt)
    return response.text

In [ ]:
def ask_rag(query,top_k=3):
    retrieved = retrieve_chunks(query,top_k=3)
    prompt=build_prompt(query,retrieved)
    answer = generate_answer_with_gemini(prompt)

    return {"query":query,"retrieved":retrieved,"prompt":prompt,"answer":answer}

In [ ]:
result = ask_rag("What is RAG?",top_k=3)

print("QUESTION\n")
print(res["query"])
print("\nANSWER")
print(res["answer"])
print("\nRETRIEVED CHUNKS")
for r in res["retrieved"]:
    print("\n")
    print("CHUNK ID",r["chunk_id"])
    print("Score:",r["score"])
    print(r["text"])